In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Any
from sklearn.metrics import precision_score#, recall_score, f1_score, roc_auc_score

from counterfactual_fraud_model import (
    OffPolicyEvaluationPipeline,
    OffPolicyEvaluationConfig,
    DataGeneratorConfig,
    LoggingPolicyConfig,
    CounterfactualEstimatorConfig,
    PipelineConfig
)

In [2]:
config = OffPolicyEvaluationConfig(
    data_generator=DataGeneratorConfig(
        # HACK: erase later
        sample_size=10_000,
        random_state=667
    ),
    logging_policy=LoggingPolicyConfig(
        cutoff=0.1,
        exploration_rate=0.05,  # Default, will be overridden in loop
        random_state=667
    ),
    counterfactual_estimator=CounterfactualEstimatorConfig(random_state=42),
    pipeline=PipelineConfig(include_data=False)
)

# Initialize pipeline with configuration
pipeline = OffPolicyEvaluationPipeline(config)


In [3]:
results = pipeline.run_pipeline(cutoff=0.1, exploration_rate=0.01, include_data=True)

In [4]:
investigate = results['data']
investigate.head()

,model_scores,is_fraud,propensity_score,model_action,policy_action
0,3.136570e-04,0,1.0,allow,allow
1,6.579500e-06,0,1.0,allow,allow
2,2.782213e-09,0,1.0,allow,allow
3,1.532518e-01,0,0.0,block,block
4,3.704167e-02,0,1.0,allow,allow


In [5]:
investigate.groupby(['model_action', 'policy_action']).size()

model_action  policy_action
allow         allow            8671
block         allow              10
              block            1319
dtype: int64

In [6]:
y_true = investigate['is_fraud']
y_pred = (investigate['model_action'] == 'block').astype(int)

precision_score(y_true, y_pred)

0.12189616252821671

In [7]:
filtered_data = investigate[investigate['policy_action'] == 'allow']

y_true = filtered_data['is_fraud']
y_pred = (filtered_data['model_action'] == 'block').astype(int)
weights = 1 / filtered_data['propensity_score']

precision_score(y_true, y_pred)

0.0

In [8]:
filtered_data[filtered_data['propensity_score'] != 1]

,model_scores,is_fraud,propensity_score,model_action,policy_action
4700,0.110508,0,0.01,block,allow
5527,0.171498,0,0.01,block,allow
5604,0.357389,0,0.01,block,allow
6829,0.110018,0,0.01,block,allow
6882,0.157852,0,0.01,block,allow
7684,0.664418,0,0.01,block,allow
7975,0.355591,0,0.01,block,allow
8752,0.186285,0,0.01,block,allow
8951,0.192204,0,0.01,block,allow
9519,0.887872,0,0.01,block,allow


In [9]:
results

{'statistics': {'total_transactions': 10000,
  'allowed_transactions': np.int64(8681),
  'blocked_transactions': np.int64(1319),
  'allow_rate': np.float64(0.8681),
  'block_rate': np.float64(0.1319),
  'fraud_rate_overall': np.float64(0.0554),
  'fraud_rate_allowed': np.float64(0.045156088008293974)},
 'ope_metrics': {'precision': {'mean': 0.0,
   'p025': 0.0,
   'p975': 0.0,
   'n_bootstrap': 5000},
  'recall': {'mean': 0.0, 'p025': 0.0, 'p975': 0.0, 'n_bootstrap': 5000},
  'fraud_rate': {'mean': 0.045187798546190275,
   'p025': 0.040895961846710197,
   'p975': 0.049609616535913316,
   'n_bootstrap': 5000},
  'average_precision': {'mean': 0.03515945088306955,
   'p025': 0.029267228499424133,
   'p975': 0.04190569503542922,
   'n_bootstrap': 5000}},
 'parameters': {'data_generator': {'alpha': 0.1,
   'beta_param': 2.0,
   'mean': -0.5,
   'sd': 0.5,
   'sample_size': 10000,
   'random_state': 667},
  'logging_policy': {'cutoff': 0.1,
   'exploration_rate': 0.01,
   'propensity_type': 

In [1]:
from counterfactual_fraud_model.config import (
    SyntheticRetrainingConfig,
    SyntheticOffPolicyEvaluationConfig,
    SyntheticDataConfig,
    ModelConfig,
    LoggingPolicyConfig,
    CounterfactualEstimatorConfig,
    RetrainingConfig,
    RetrainingModelConfig,
    PipelineConfig,
    ModelType,
    RetrainingStrategy
)

# Import the pipeline
from counterfactual_fraud_model.pipelines import SyntheticRetrainingPipeline

In [2]:
config = SyntheticRetrainingConfig(
    base_config=SyntheticOffPolicyEvaluationConfig(
        synthetic_data=SyntheticDataConfig(
            n_samples=5_000,  # Small for quick testing
            n_features=10,
            n_informative=6,
            n_redundant=2,  # Ensure sum doesn't exceed n_features
            n_repeated=0,   # Keep it simple
            random_state=42
        ),
        model=ModelConfig(
            model_type=ModelType.LIGHTGBM,
            random_state=42
        ),
        logging_policy=LoggingPolicyConfig(
            cutoff=0.05,
            exploration_rate=0.1,
            random_state=42
        ),
        counterfactual_estimator=CounterfactualEstimatorConfig(
            n_bootstrap=500,  # Reduced for speed
            random_state=42
        )
    ),
    retraining=RetrainingConfig(
        retrain_test_size=0.3,
        retrain_model=RetrainingModelConfig(
            base_model=ModelConfig(
                model_type=ModelType.LIGHTGBM,
                random_state=42
            ),
            strategy=RetrainingStrategy.FILTERING,
            classification_threshold=0.1
        )
    )
)

pipeline = SyntheticRetrainingPipeline(config)

In [3]:
pipeline.generate_logging_policy_data(
    logging_policy_cutoff=0.05,
    logging_policy_exploration_rate=0.1
)

# Test getter methods
original_results = pipeline.get_original_results()
train_data = pipeline.get_train_policy_data()
test_data = pipeline.get_test_policy_data()

In [4]:
test_data[test_data.propensity_score < 1]

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,is_fraud,model_scores,propensity_score,model_action,policy_action
2381,-0.238727,-0.382924,0.953310,0.492325,1.857344,0.630523,2.689564,-3.473522,-1.440981,2.233747,0,0.089924,0.0,block,block
764,-2.004418,0.965653,-0.552069,0.867929,1.220186,0.558746,4.078713,-0.890921,1.152473,2.199349,1,0.050376,0.0,block,block
1017,1.188811,-0.061211,0.738055,0.063767,-1.312756,-0.571942,-2.405827,-2.893997,-0.811600,-1.800615,0,0.147021,0.1,block,allow
2245,-0.477623,0.115938,0.922038,0.109399,2.730187,0.685308,3.130902,-2.265335,-0.806557,2.094464,0,0.152783,0.0,block,block
2422,-3.140672,0.095804,0.180236,-1.062917,-0.309693,6.227987,-0.975829,3.388169,-0.113309,1.997149,0,0.369712,0.0,block,block
854,-1.302853,-0.781638,0.122337,0.001580,-1.192055,1.721645,0.080422,0.609130,0.649480,1.967363,0,0.177402,0.0,block,block
1115,-1.398780,-1.668441,0.849491,-0.683538,-1.850378,5.435321,-3.055081,2.282666,0.989130,1.618965,0,0.177142,0.0,block,block
950,-2.833656,0.230541,-0.000157,0.440310,-4.129691,0.336123,0.217177,-0.447912,2.656728,1.954478,0,0.185948,0.0,block,block


In [7]:
results = pipeline.run_retrain_pipeline(retraining_config=RetrainingConfig(
            retrain_test_size=0.3,
            retrain_model=RetrainingModelConfig(
                base_model=ModelConfig(
                    model_type=ModelType.LIGHTGBM,
                    random_state=42
                ),
                strategy=RetrainingStrategy.FILTERING,
                classification_threshold=0.1
            )
        ))

In [8]:
results

{'original_results': {'model_performance': {'precision': 0.34615384615384615,
   'recall': 0.18,
   'f1': 0.23684210526315788,
   'roc_auc': 0.8143346938775511,
   'average_precision': 0.1940385958239888},
  'dataset_info': {'n_samples': 5000,
   'n_features': 10,
   'n_informative': 6,
   'fraud_rate': np.float64(0.0202),
   'n_fraud': np.int64(101),
   'n_legitimate': np.int64(4899),
   'class_balance': [0.985, 0.015]}},
 'retrained_model_performance': {'precision': 0.3333333333333333,
  'recall': 0.06666666666666667,
  'f1': 0.1111111111111111,
  'roc_auc': 0.6464399092970521,
  'average_precision': 0.1339143511077216},
 'ope_metrics': {'precision': {'mean': 0.30816746031746034,
   'p025': 0.0,
   'p975': 1.0,
   'n_bootstrap': 500},
  'recall': {'mean': 0.06987586853438112,
   'p025': 0.0,
   'p975': 0.24086538461538418,
   'n_bootstrap': 500},
  'fraud_rate': {'mean': 0.017170144374868128,
   'p025': 0.008289004551343047,
   'p975': 0.02774365302997461,
   'n_bootstrap': 500},
  '